In [64]:
import pandas as pd
from pathlib import Path    
from statsmodels.stats.multitest import multipletests
from scipy import stats
import numpy as np

In [65]:
result_path = Path("../thesis_results/downstream_task")
t_test_methods_list = ["mrs-forest", "fw-mrs-temperature", "fw-mrs-temperature-svm"]
# bias_types = ["less_negative_class", "less_positive_class", "mean_difference"]
bias_types = ["mean_difference"]
metrics = ["AUROC", "AUPRC"]
less_bias_strengths = ["0.1"]
mean_bias_strengthts = ["0.8"]
datasets = ["breast_cancer", "folktables_employment", "folktables_income", "hr_analytics", "loan_prediction"]

In [66]:
method_pairs = []
for j in range(1, len(t_test_methods_list)):
    method_pairs.append((t_test_methods_list[0], t_test_methods_list[j]))
method_pairs

[('mrs-forest', 'fw-mrs-temperature'),
 ('mrs-forest', 'fw-mrs-temperature-svm')]

In [67]:
aurocs = []
auprcs = []
dict_list = []
for dataset in datasets:
    for bias_type in bias_types:
        for method in t_test_methods_list:
            if bias_type == "mean_difference":
                bias_strengths = mean_bias_strengthts
            else : 
                bias_strengths = less_bias_strengths
            for bias_strength in bias_strengths:
                json_directory = result_path / dataset / bias_type /  bias_strength/ method / "classification_results"
                auroc_file = pd.read_json(str(json_directory / "rf_auroc.json"))
                auprc_file = pd.read_json(str(json_directory / "rf_auprc.json"))
                dict_list.append({"Method": method, "Data Set": dataset, "AUROC": auroc_file.values, "AUPRC":auprc_file.values, 
                                  "Bias Type": bias_type, "Bias Strength": bias_strength})
result_df = pd.DataFrame(data=dict_list)

In [68]:
result_df.explode(["AUROC", "AUPRC"])

,Method,Data Set,AUROC,AUPRC,Bias Type,Bias Strength
0,mrs-forest,breast_cancer,[0.994616104868913],[0.997167138114538],mean_difference,0.8
0,mrs-forest,breast_cancer,[0.9936797752808981],[0.9970514811818301],mean_difference,0.8
0,mrs-forest,breast_cancer,[0.9927434456928831],[0.9967839163868871],mean_difference,0.8
0,mrs-forest,breast_cancer,[0.982787473105426],[0.9928351939406431],mean_difference,0.8
0,mrs-forest,breast_cancer,[0.9940814393939391],[0.997164775000909],mean_difference,0.8
...,...,...,...,...,...,...
14,fw-mrs-temperature-svm,loan_prediction,[0.7196969696969691],[0.8236220308248771],mean_difference,0.8
14,fw-mrs-temperature-svm,loan_prediction,[0.687626262626262],[0.8250722632261601],mean_difference,0.8
14,fw-mrs-temperature-svm,loan_prediction,[0.6785353535353531],[0.792282004016116],mean_difference,0.8
14,fw-mrs-temperature-svm,loan_prediction,[0.719763252702007],[0.841636204492692],mean_difference,0.8


In [69]:
def corrected_t_test(first_values, second_values, n_folds=5.0):
    differences = first_values - second_values
    mean_differences = np.mean(differences)
    var_differences = np.var(differences)
    train_size = n_folds - 1.0
    test_size = 1.0 
    correction_factor = (1.0 / len(differences)) + (test_size / train_size)
    return mean_differences / (np.sqrt(correction_factor * var_differences))

In [70]:
p_values = []
for dataset in datasets:
    for bias_type in bias_types:
            for bias_strength in bias_strengths:
                for first_method_name, second_metric_name in method_pairs:
                    for metric in metrics:
                        first_metrics = result_df.loc[(result_df["Method"]==first_method_name) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & (result_df["Data Set"]==dataset)][metric].values[0]
                        second_metrics = result_df.loc[(result_df["Method"]==second_metric_name) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & (result_df["Data Set"]==dataset)][metric].values[0]
                        t_statistic = corrected_t_test(np.squeeze(first_metrics), np.squeeze(second_metrics))
                        p_values.append(stats.t.sf(np.abs(t_statistic), len(first_metrics-1)) * 2)
corrected_p_values = multipletests(p_values, method="fdr_bh")

In [71]:
i = 0
for dataset in datasets:
    for bias_type in bias_types:
            for bias_strength in bias_strengths:
                for first_method_name, second_metric_name in method_pairs:
                    for metric in metrics:
                        print(f"p value for {metric}, {dataset}, {bias_type}, {bias_strength}, {first_method_name},\
{second_metric_name} is: {corrected_p_values[0][i]}")
                        i += 1

p value for AUROC, breast_cancer, mean_difference, 0.8, mrs-forest,fw-mrs-temperature is: False
p value for AUPRC, breast_cancer, mean_difference, 0.8, mrs-forest,fw-mrs-temperature is: False
p value for AUROC, breast_cancer, mean_difference, 0.8, mrs-forest,fw-mrs-temperature-svm is: False
p value for AUPRC, breast_cancer, mean_difference, 0.8, mrs-forest,fw-mrs-temperature-svm is: False
p value for AUROC, folktables_employment, mean_difference, 0.8, mrs-forest,fw-mrs-temperature is: True
p value for AUPRC, folktables_employment, mean_difference, 0.8, mrs-forest,fw-mrs-temperature is: True
p value for AUROC, folktables_employment, mean_difference, 0.8, mrs-forest,fw-mrs-temperature-svm is: False
p value for AUPRC, folktables_employment, mean_difference, 0.8, mrs-forest,fw-mrs-temperature-svm is: False
p value for AUROC, folktables_income, mean_difference, 0.8, mrs-forest,fw-mrs-temperature is: False
p value for AUPRC, folktables_income, mean_difference, 0.8, mrs-forest,fw-mrs-temperat